# Atividade 5 — IDHM por estado: melt + matplotlib

Base: `Tabela4.csv` (IDHM por Unidade da Federação), na mesma pasta.

In [ ]:
# pip install pandas matplotlib
import pandas as pd
import matplotlib.pyplot as plt

## 1. Abrir o CSV com `;` e vírgula decimal

In [ ]:
def ler_csv(arquivo, **kw):
    """CSV do IBGE: separador ; e vírgula decimal. Tenta utf-8 e depois latin-1."""
    try:
        return pd.read_csv(arquivo, sep=";", decimal=",", encoding="utf-8", **kw)
    except UnicodeDecodeError:
        return pd.read_csv(arquivo, sep=";", decimal=",", encoding="latin-1", **kw)

In [ ]:
# a 1ª linha é só o título da tabela
df = ler_csv("Tabela4.csv", skiprows=1)
df.head()

## 2. Limpar colunas vazias

In [ ]:
df = df.dropna(axis=1, how="all")            # colunas totalmente vazias
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df = df.dropna(subset=["Sigla"])
df.columns = [str(c).strip() for c in df.columns]
print(df.shape)
df.head()

## 3. Estados ordenados pelo maior IDH em 2024

In [ ]:
ranking_2024 = df.sort_values("2024", ascending=False)[["Sigla", "Estado", "2024"]].reset_index(drop=True)
ranking_2024

## 4. Maior melhora de IDH entre 1991 e 2024

In [ ]:
df["melhora_1991_2024"] = (df["2024"] - df["1991"]).round(3)
melhor = df.sort_values("melhora_1991_2024", ascending=False).iloc[0]
print(f"Maior melhora: {melhor['Estado']} ({melhor['Sigla']}) -> +{melhor['melhora_1991_2024']}")
df.sort_values("melhora_1991_2024", ascending=False)[["Estado", "1991", "2024", "melhora_1991_2024"]].head()

## 5. Existe algum estado em que o IDH piorou?

In [ ]:
piorou = df[df["2024"] < df["1991"]]
print("Estados com IDH 2024 menor que em 1991:", list(piorou["Estado"]) or "nenhum")

# olhando ano a ano: em quais estados houve queda de um ano para o seguinte?
anos = [c for c in df.columns if str(c).isdigit()]
quedas = []
for _, linha in df.iterrows():
    for a, b in zip(anos, anos[1:]):
        if pd.notna(linha[a]) and pd.notna(linha[b]) and linha[b] < linha[a]:
            quedas.append((linha["Sigla"], f"{a}->{b}", round(linha[b] - linha[a], 3)))
quedas = pd.DataFrame(quedas, columns=["Sigla", "periodo", "variacao"])
print("Quedas por período (quantos estados caíram):")
quedas["periodo"].value_counts()

## 6. Formato largo -> longo (`melt`)

In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

id_vars = ["Sigla", "Código", "Estado"]
df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo.head()

## 7. Apenas Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mg["Ano"], mg["IDH"], marker="o", color="darkgreen")
ax.set_title("Evolução do IDH - Minas Gerais")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
fig.tight_layout()
plt.show()

## 8. Evolução do IDH de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()